## Model Inference

This notebook is designed to load the pre-trained XGBoost model, perform inference on data for a specified snapshot_date, and save the prediction results to the gold table for downstream analysis or production use. moreover, it also conduct a backfiill process.

In [367]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

import model_inference

In [368]:
# Build a .py script that takes a snapshot date, loads a model artefact and make an inference and save to datamart

## set up pyspark session

In [369]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

## set up config

In [370]:
# set up config
model_train_date_str = "2024-09-01"
snapshot_date_str = "2024-01-01"
model_name = "credit_model_2024_09_01.pkl"
train_test_period_months = 12
oot_period_months = 2
train_test_ratio = 0.8
config = {}
config["snapshot_date_str"] = snapshot_date_str
config["snapshot_date"] = datetime.strptime(config["snapshot_date_str"], "%Y-%m-%d")
config["model_name"] = model_name
config["model_bank_directory"] = "model_bank/"
config["model_artefact_filepath"] = config["model_bank_directory"] + config["model_name"]
config["model_train_date_str"] = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"] =  oot_period_months
config["model_train_date"] =  datetime.strptime(model_train_date_str, "%Y-%m-%d")
config["oot_end_date"] =  config['model_train_date'] - timedelta(days = 1)
config["oot_start_date"] =  config['model_train_date'] - relativedelta(months = oot_period_months)
config["train_test_end_date"] =  config["oot_start_date"] - timedelta(days = 1)
config["train_test_start_date"] =  config["oot_start_date"] - relativedelta(months = train_test_period_months)
config["train_test_ratio"] = train_test_ratio 

## load model artefact from model bank

In [371]:
# Load the model from the pickle file
with open(config["model_artefact_filepath"], 'rb') as file:
    model_artefact = pickle.load(file)

print("Model loaded successfully! " + config["model_artefact_filepath"])

Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


## load and process data for model inference

In [372]:
def read_gold_table(table, gold_db, spark):
    """
    Helper function to read all partitions of a gold table
    """
    folder_path = os.path.join(gold_db, table)
    files_list = [os.path.join(folder_path, os.path.basename(f)) for f in glob.glob(os.path.join(folder_path, '*'))]
    df = spark.read.option("header", "true").parquet(*files_list)
    return df

In [373]:
X_spark = read_gold_table('feature_store', 'datamart/gold', spark)
y_spark = read_gold_table('label_store', 'datamart/gold', spark)
X_df = X_spark.toPandas().sort_values(by='customer_id')
y_df = y_spark.toPandas().sort_values(by='customer_id')

In [374]:
# Consider data from model training date
# Makesure snapshot_date tyoes are the same
y_df['snapshot_date'] = pd.to_datetime(y_df['snapshot_date'])
X_df['snapshot_date'] = pd.to_datetime(X_df['snapshot_date'])

y_model_df = y_df[(y_df['snapshot_date'] >= config['train_test_start_date']) & (y_df['snapshot_date'] <= config['model_train_date'])]
X_model_df = X_df[np.isin(X_df['customer_id'], y_model_df['customer_id'].unique())]

# Create OOT split
y_oot = y_model_df[(y_model_df['snapshot_date'] >= config['oot_start_date']) & (y_model_df['snapshot_date'] <= config['oot_end_date'])]
X_oot = X_model_df[np.isin(X_model_df['customer_id'], y_oot['customer_id'].unique())]

# Everything else goes into train-test
y_traintest = y_model_df[y_model_df['snapshot_date'] <= config['train_test_end_date']]
X_traintest = X_model_df[np.isin(X_model_df['customer_id'], y_traintest['customer_id'].unique())]

In [387]:
X_train, X_test, y_train, y_test = train_test_split(X_traintest, y_traintest, 
                                                    test_size=config['train_test_ratio'], 
                                                    random_state=88, 
                                                    shuffle=True, 
                                                    stratify=y_traintest['label'])

print('X_test', X_test.shape[0])
print('y_test', y_test.shape[0], round(y_test['label'].mean(), 2))

X_test

X_test 4767
y_test 4767 0.28


,customer_id,snapshot_date,age,annual_income,monthly_inhand_salary,num_bank_accounts,num_credit_card,interest_rate,num_of_loan,delay_from_due_date,...,avg_fe_11,avg_fe_12,avg_fe_13,avg_fe_14,avg_fe_15,avg_fe_16,avg_fe_17,avg_fe_18,avg_fe_19,avg_fe_20
2190,CUS_0x3048,2023-12-01,40,81093.156250,7017.763184,10.0,7.0,17.0,7.00000,29,...,102.250000,120.333333,107.916667,122.000000,164.500000,94.416667,78.750000,68.166667,88.416667,134.583333
8365,CUS_0x269d,2023-12-01,39,57810.679688,4896.556641,7.0,5.0,7.0,4.00000,29,...,116.833333,103.750000,70.500000,70.500000,116.416667,105.416667,120.750000,94.166667,125.166667,186.750000
7297,CUS_0xbab8,2023-11-01,48,175972.000000,14771.333008,3.0,4.0,3.0,2.00000,2,...,91.363636,72.090909,50.909091,110.545455,67.545455,68.727273,46.818182,110.272727,113.636364,85.181818
3357,CUS_0x74f5,2023-08-01,14,74733.437500,5229.153320,9.0,6.0,19.0,5.00000,17,...,123.500000,55.250000,110.625000,78.500000,120.000000,81.250000,124.250000,64.250000,76.000000,42.250000
2024,CUS_0x4534,2023-09-01,37,20496.789062,1635.065796,3.0,3.0,6.0,2.00000,20,...,207.666667,80.666667,186.888889,109.444444,92.777778,98.777778,95.666667,60.888889,166.333333,82.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6693,CUS_0x2b51,2023-04-01,26,47456.320312,4007.693359,10.0,3.0,10.0,3.52656,11,...,87.500000,37.000000,122.250000,72.000000,129.500000,87.500000,132.250000,165.500000,163.000000,124.750000
5607,CUS_0x6d94,2023-10-01,44,9844.974609,824.414612,10.0,7.0,19.0,0.00000,7,...,108.200000,102.600000,58.700000,129.300000,64.500000,66.300000,132.200000,113.600000,133.500000,143.100000
3403,CUS_0xa40b,2023-08-01,39,31530.759766,2676.563232,1.0,3.0,1.0,4.00000,6,...,104.875000,127.125000,81.750000,98.875000,43.250000,79.000000,79.000000,83.625000,43.125000,72.250000
2081,CUS_0x87f0,2023-09-01,43,44964.218750,3898.018311,6.0,10.0,15.0,3.00000,27,...,75.777778,52.222222,80.777778,105.555556,111.111111,105.555556,98.333333,84.555556,77.222222,105.444444


In [388]:
# Transform data into numpy arrays
X_train_arr = X_train.drop(columns=['customer_id', 'snapshot_date']).values
X_test_arr = X_test.drop(columns=['customer_id', 'snapshot_date']).values
X_oot_arr = X_oot.drop(columns=['customer_id', 'snapshot_date']).values

y_train_arr = y_train['label'].values
y_test_arr = y_test['label'].values
y_oot_arr = y_oot['label'].values

In [389]:
transformer_stdscaler = model_artefact['preprocessing_transformers']['stdscaler']


In [396]:
X_test_df = X_test.copy()
X_test_ordered = X_test_df[model_artefact["feature_columns"]]
X_test_arr = X_test_ordered.values

In [390]:
X_inference = model_artefact['preprocessing_transformers']['stdscaler'].transform(X_test_arr)
print(X_test_arr.shape)
print(X_test_arr.dtype)

(4767, 72)
float64


## XGBoost prediction inference

In [391]:
# load model
model = model_artefact["model"]

# predict model
y_inference = model.predict_proba(X_inference)[:, 1]

# output
y_inference_pdf = X_test_df[["customer_id", "snapshot_date"]].copy()
y_inference_pdf["model_name"] = model_artefact["model_version"]
y_inference_pdf["model_predictions"] = y_inference

y_inference_pdf.head()

,customer_id,snapshot_date,model_name,model_predictions
2190,CUS_0x3048,2023-12-01,credit_model_2024_09_01,0.622699
8365,CUS_0x269d,2023-12-01,credit_model_2024_09_01,0.136337
7297,CUS_0xbab8,2023-11-01,credit_model_2024_09_01,0.209850
3357,CUS_0x74f5,2023-08-01,credit_model_2024_09_01,0.496724
2024,CUS_0x4534,2023-09-01,credit_model_2024_09_01,0.210344


## Save model inference to datamart gold table

In [392]:
# create bronze datalake
gold_directory = f"datamart/gold/model_predictions/{config["model_name"][:-4]}/"
print(gold_directory)

if not os.path.exists(gold_directory):
    os.makedirs(gold_directory)

# save gold table - IRL connect to database to write
partition_name = config["model_name"][:-4] + "_predictions_" + snapshot_date_str.replace('-','_') + '.parquet'
filepath = gold_directory + partition_name
spark.createDataFrame(y_inference_pdf).write.mode("overwrite").parquet(filepath)
# df.toPandas().to_parquet(filepath,
#           compression='gzip')
print('saved to:', filepath)

datamart/gold/model_predictions/credit_model_2024_09_01/
saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_01_01.parquet


## backfill

In [393]:
# set up config
snapshot_date_str = "2023-01-01"
start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

In [394]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)

In [395]:
for snapshot_date in dates_str_lst:
    print(snapshot_date)
    model_inference.main(snapshot_date, model_name)

2023-01-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 1, 1, 0, 0),
 'snapshot_date_str': '2023-01-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 8974 2023-01-01 00:00:00


/usr/local/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


ValueError: X has 20 features, but StandardScaler is expecting 72 features as input.

In [364]:
## Check datamart

In [365]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [366]:
folder_path = "datamart/gold/model_predictions/credit_model_2024_09_01/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",df.count())

df.show()

row_count: 14301
+-----------+-------------------+--------------------+-------------------+
|customer_id|      snapshot_date|          model_name|  model_predictions|
+-----------+-------------------+--------------------+-------------------+
| CUS_0x8442|2023-12-01 00:00:00|credit_model_2024...|0.08619970828294754|
| CUS_0x311a|2023-12-01 00:00:00|credit_model_2024...|0.13653558492660522|
| CUS_0x517c|2023-05-01 00:00:00|credit_model_2024...|0.10147149860858917|
| CUS_0x78e0|2023-05-01 00:00:00|credit_model_2024...|0.02472226321697235|
| CUS_0x9074|2023-10-01 00:00:00|credit_model_2024...|0.04409293457865715|
| CUS_0x295f|2023-05-01 00:00:00|credit_model_2024...|0.04413345828652382|
| CUS_0x12a9|2023-05-01 00:00:00|credit_model_2024...|0.07540223002433777|
|  CUS_0xf50|2023-11-01 00:00:00|credit_model_2024...|0.09351382404565811|
| CUS_0x65fc|2023-12-01 00:00:00|credit_model_2024...|0.02623230591416359|
| CUS_0x72d2|2023-08-01 00:00:00|credit_model_2024...|0.46622616052627563|
| CUS_0x